In [1]:
from datetime import date
from pathlib import Path

from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from evaluation.flights import evaluation
from flights.flights_match_adsb import get_matching_icaos_in_flights
from flights_ai_model.features import get_icaos_in_features, read_icaos_from_features
from flights_ai_model.inference import inference
from flights_ai_model.train import train
from flights_ai_model.training.training_days import sequence_training_days
import polars as pl
from flights_ai_model.validation import flights_df_comparison

In [2]:
val_date = date(2026,3,1)
sfdps_flights = get_sfdps_flights_day(val_date)
icaos = sfdps_flights.get_column("icao").to_list()
df_adsb = read_icaos_from_adsb(val_date, icaos)
df_gold = get_matching_icaos_in_flights(sfdps_flights, use_readsb_parquet=True, target_date=val_date)
icaos = list(set(df_gold.get_column("icao")))

In [ ]:
# from data_engineering.utils import OUTPUT_DIR
# df_gold.write_parquet(OUTPUT_DIR / "tmp/df_gold.parquet")

In [ ]:
# from data_engineering.utils import OUTPUT_DIR
# df_gold = pl.read_parquet(OUTPUT_DIR / "tmp/df_gold.parquet")
# icaos = list(set(df_gold.get_column("icao")))

In [3]:
from datetime import date
from flights_algorithm.main import run_main_2
df_test = run_main_2(val_date, icaos=icaos)

In [4]:
df_gold.sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a008c9""","""DAL2519""","""N101DU""",2026-03-01 00:01:00,"""KRDU""",2026-03-01 02:35:00,"""KMSP""",false,false,false,false,"""BCS1""","""BANK OF UTAH TRUSTEE""","""AIRBUS A220-100""",""""""
"""a0fea3""","""UPS5704""","""N163UP""",2026-03-01 00:01:00,"""KMEM""",2026-03-01 00:52:00,"""KMSY""",false,false,false,false,"""A306""","""UNITED PARCEL SERVICE CO""","""AIRBUS A-300-600""",""""""
"""a495e9""","""FFT1565""","""N395FR""",2026-03-01 00:01:00,"""KMCO""",2026-03-01 01:45:00,"""KCVG""",false,false,false,false,"""A20N""","""BANK OF UTAH TRUSTEE""","""AIRBUS A-320neo""",""""""
"""a54f67""","""N441M""","""N441M""",2026-03-01 00:01:00,"""KVES""",2026-03-01 00:42:00,"""KLOU""",false,false,false,false,"""C441""","""COVINGTON HOLDINGS INC""","""CESSNA 441 Conquest""",""""""
"""a58aba""","""SGX456""","""N456ZW""",2026-03-01 00:01:00,"""KOQU""",2026-03-01 01:14:00,"""KIAD""",false,false,false,false,"""CRJ2""","""AIR WISCONSIN AIRLINES LLC""","""BOMBARDIER Regional Jet CRJ-20…",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""abbac4""","""JRE855""","""N855JS""",2026-03-01 23:23:00,"""KAPF""",2026-03-01 23:56:00,"""KVRB""",false,false,false,false,"""C56X""","""FE 5064 LLC""","""CESSNA 560XL Citation XLS""",""""""
"""ad8c37""","""N972MS""","""N972MS""",2026-03-01 23:27:00,"""KPBI""",2026-03-01 23:48:00,"""KFXE""",false,true,false,false,"""GLF4""","""GLF4 1285 LLC""","""GULFSTREAM 4""",""""""
"""a6d6ec""","""N54PV""","""N54PV""",2026-03-01 23:28:00,"""KLGB""",2026-03-01 23:50:00,"""KCMA""",false,false,false,false,"""C510""","""NOSTROMO 2021 LLC""","""CESSNA 510 Citation Mustang""","""A1"""


In [ ]:
# df_test = df_test.filter(pl.col("landing_time").dt.date() == date(2026,2,1))

In [19]:
from data_engineering.utils import OUTPUT_DIR
df_test.write_parquet(OUTPUT_DIR/"tmp/df_test_0.parquet")

In [5]:
from data_engineering.utils import OUTPUT_DIR
df_test = pl.read_parquet(OUTPUT_DIR/"tmp/df_test_0.parquet")

In [14]:
df_gold

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""046c56""","""LN90LJ""","""""",2026-03-01 13:03:00,"""TBPB""",2026-03-01 22:14:00,"""KTMB""",false,false,false,false,"""""","""""","""""","""A2"""
"""0a805a""","""FSS301""","""C6-TAL""",2026-03-01 13:17:00,"""MYNN""",2026-03-01 14:34:00,"""KFXE""",false,false,false,false,"""C208""","""""","""CESSNA 208 Caravan""",""""""
"""0c43b7""","""BOE054""","""""",2026-03-01 19:59:00,"""KBFI""",2026-03-01 21:04:00,"""KBFI""",false,false,false,false,"""""","""""","""""",""""""
"""0d0cf3""","""XADLN""","""XA-DLN""",2026-03-01 14:42:00,"""MDPC""",2026-03-01 21:00:00,"""KFLL""",false,true,false,false,"""H25B""","""""","""HAWKER BEECHCRAFT Hawker 750/8…","""A2"""
"""0d11e8""","""BOE242""","""""",2026-03-01 21:06:00,"""KRNT""",2026-03-01 22:34:00,"""KPDX""",false,false,false,false,"""""","""""","""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""c022e9""","""DCM2092""","""C-FNFS""",2026-03-01 22:22:00,"""KOPF""",2026-03-01 23:19:00,"""KSGJ""",false,true,false,false,"""C25B""","""Stampede Aviation Inc.""","""CESSNA 525B Citation CJ3""",""""""
"""c027ae""","""CFPAR""","""C-FPAR""",2026-03-01 20:45:00,"""KFXE""",2026-03-01 23:12:00,"""KHOU""",false,false,false,false,"""CL35""","""AVIATION CMP INC.""","""BOMBARDIER BD-100 Challenger 3…",""""""
"""c031ba""","""ASP716""","""C-FSVP""",2026-03-01 04:48:00,"""KSDL""",2026-03-01 05:36:00,"""KLAS""",false,false,false,false,"""DH3T""","""""","""DE HAVILLAND DHC-3 Turbo Otter""",""""""


In [13]:
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
df_gold = df_gold.filter(pl.col("icao").is_in(df_test.get_column("icao")))


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_39168/3753629309.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_39168/3753629309.py:2: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_gold = df_gold.filter(pl.col("icao").is_in(df_test.get_column("icao")))


In [30]:
df_gold = df_gold.filter(pl.col("landing_time").dt.date() == val_date).filter(pl.col("icao").is_in(df_test.get_column("icao").implode()))

In [5]:
df, stats = flights_df_comparison(df_test, df_gold)


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 3593
Predicted flights: 7288  |  Gold flights: 7289
Correctly matched: 7150  |  Predicted unmatched: 138  |  Gold flights missed: 139
Precision: 98.11%  |  Recall: 98.09%  |  F1: 98.10%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 0/7150 matched (0.00%)
landing_airport_ident: 0/7150 matched (0.00%)
overall: 0/14300 matched (0.00%)


In [6]:
COLUMNS = ["icao","takeoff_time", "landing_time", "takeoff_time_df2", "landing_time_df2", "takeoff_airport_ident_df2", "landing_airport_ident_df2", "match_status"]

In [7]:
df = df.select(COLUMNS)
df.filter(pl.col("match_status") == "df2_only")

icao,takeoff_time,landing_time,takeoff_time_df2,landing_time_df2,takeoff_airport_ident_df2,landing_airport_ident_df2,match_status
str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str
"""a02b88""",null,null,2026-03-01 17:03:00,2026-03-01 17:48:00,"""KGPT""","""KCSG""","""df2_only"""
"""a02b88""",null,null,2026-03-01 18:09:00,2026-03-01 18:38:00,"""KCSG""","""KBHM""","""df2_only"""
"""a05342""",null,null,2026-03-01 18:58:00,2026-03-01 19:54:00,"""KSKF""","""KLNC""","""df2_only"""
"""a05342""",null,null,2026-03-01 20:08:00,2026-03-01 21:00:00,"""KLNC""","""KSGR""","""df2_only"""
"""a06699""",null,null,2026-03-01 03:38:00,2026-03-01 04:41:00,"""KSFO""","""KTRM""","""df2_only"""
…,…,…,…,…,…,…,…
"""ad7e2e""",null,null,2026-03-01 15:51:00,2026-03-01 16:45:00,"""KMLI""","""KHYR""","""df2_only"""
"""ad7e2e""",null,null,2026-03-01 17:05:00,2026-03-01 17:51:00,"""KHYR""","""KMLI""","""df2_only"""
"""ad92cf""",null,null,2026-03-01 00:03:00,2026-03-01 00:55:00,"""KELP""","""KIWA""","""df2_only"""


In [9]:
df.filter(pl.col("icao") =="a05342")

icao,takeoff_time,landing_time,takeoff_time_df2,landing_time_df2,takeoff_airport_ident_df2,landing_airport_ident_df2,match_status
str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str
"""a05342""",2026-03-01 11:49:13.866,2026-03-01 12:38:51.395,2026-03-01 11:49:00,2026-03-01 12:38:00,"""KSGR""","""KLNC""","""matched"""
"""a05342""",2026-03-01 13:12:13.402,2026-03-01 14:07:49.833,2026-03-01 13:12:00,2026-03-01 14:11:00,"""KLNC""","""KSKF""","""matched"""
"""a05342""",2026-03-01 18:58:42.529,2026-03-01 20:57:58.793,null,null,null,null,"""df1_only"""
"""a05342""",null,null,2026-03-01 18:58:00,2026-03-01 19:54:00,"""KSKF""","""KLNC""","""df2_only"""
"""a05342""",null,null,2026-03-01 20:08:00,2026-03-01 21:00:00,"""KLNC""","""KSGR""","""df2_only"""


In [7]:
df_gold.sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a08df7""","""KAP9""","""N135CA""",2026-03-01 00:00:00,"""KSDY""",2026-03-01 01:26:00,"""KBIL""",false,false,false,false,"""P212""","""HYANNIS AIR SERVICE INC""","""TECNAM P-2012 Traveller""","""A1"""
"""a1a0df""","""N204A""","""N204A""",2026-03-01 00:00:00,"""KPSP""",2026-03-01 00:38:00,"""KVNY""",false,true,false,false,"""GLF5""","""MERCURY INTERNATIONAL HOLDINGS…","""GULFSTREAM 5""","""A3"""
"""a2ff72""","""N2921Q""","""N2921Q""",2026-03-01 00:00:00,"""HAR""",2026-03-01 00:31:00,"""LOM""",false,false,false,false,"""P32R""","""AMBLER AVIATION LLC""","""PIPER PA-32R-300/301/301T""","""A1"""
"""a3a160""","""MVJ33""","""N333EH""",2026-03-01 00:00:00,"""KBZN""",2026-03-01 02:06:00,"""KNUQ""",false,true,false,false,"""GLF5""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""GULFSTREAM 5""","""A3"""
"""a6eed5""","""GJS4497""","""N546GJ""",2026-03-01 00:00:00,"""KORD""",2026-03-01 00:52:00,"""KLEX""",false,false,false,false,"""CRJ7""","""SKYWEST AIRLINES INC""","""BOMBARDIER Regional Jet CRJ-70…","""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a6eed5""","""GJS4573""","""N546GJ""",2026-03-01 23:25:00,"""KMLI""",2026-03-01 23:55:00,"""KORD""",false,false,false,false,"""CRJ7""","""SKYWEST AIRLINES INC""","""BOMBARDIER Regional Jet CRJ-70…","""A2"""
"""ad8c37""","""N972MS""","""N972MS""",2026-03-01 23:27:00,"""KPBI""",2026-03-01 23:48:00,"""KFXE""",false,true,false,false,"""GLF4""","""GLF4 1285 LLC""","""GULFSTREAM 4""","""A2"""
"""a6d6ec""","""N54PV""","""N54PV""",2026-03-01 23:28:00,"""KLGB""",2026-03-01 23:50:00,"""KCMA""",false,false,false,false,"""C510""","""NOSTROMO 2021 LLC""","""CESSNA 510 Citation Mustang""","""A1"""


In [15]:
df.filter(pl.col("match_status") == "df1_only")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""046c56""","""""","""""",2026-03-01 13:03:38.420,"""""",2026-03-01 13:52:33.460,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 14:32:46.860,"""""",2026-03-01 16:24:12.920,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 18:41:27.650,"""""",2026-03-01 19:59:43.450,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 20:43:21.070,"""""",2026-03-01 21:00:47.270,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 21:38:18.270,"""""",2026-03-01 22:11:18.120,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""add7dc""","""""","""""",2026-03-01 00:52:21.670,"""""",2026-03-01 01:23:45.780,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""ade27a""","""""","""""",2026-03-01 17:21:18.040,"""""",2026-03-01 18:35:55.530,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""ade27a""","""""","""""",2026-03-01 19:42:52.710,"""""",2026-03-01 20:18:26.480,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""


In [13]:
df.filter(pl.col("icao") == "c06401")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""c06401""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CGLWQ""","""C-GLWQ""",2026-03-01 19:49:00,"""KMKE""",2026-03-01 20:12:00,"""KGRR""",false,false,false,false,"""P180""","""Flightpath Charter Airways Inc…","""PIAGGIO P-180 Avanti""","""A2""",null,null,"""df2_only"""


In [14]:
df_2 = read_icaos_from_adsb(date(2026,3,1), icaos=["c06401"])
df_2

time,icao,registration,callsign,lat,lon,baro_altitude_ft,on_ground,ground_speed_kt,track_deg,aircraft_type,aircraft_description,owner,category,pia,ladd,military,interesting
datetime[ms],str,str,str,f64,f64,i32,bool,f32,f32,str,str,str,str,bool,bool,bool,bool
2026-03-01 19:49:18.360,"""c06401""","""C-GLWQ""","""CGLWQ""",42.943695,-87.895333,425,false,152.300003,7.5,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:19.370,"""c06401""","""C-GLWQ""",null,42.944458,-87.89516,500,false,152.300003,7.5,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:19.860,"""c06401""","""C-GLWQ""",null,42.944824,-87.895096,525,false,153.399994,7.9,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:20.670,"""c06401""","""C-GLWQ""",null,42.945419,-87.894968,600,false,154.600006,8.2,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:22.290,"""c06401""","""C-GLWQ""","""CGLWQ""",42.946518,-87.894777,700,false,155.600006,8.1,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-01 20:07:50.360,"""c06401""","""C-GLWQ""",null,42.875216,-85.555115,500,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 20:07:50.910,"""c06401""","""C-GLWQ""",null,42.875244,-85.554753,500,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 20:07:51.820,"""c06401""","""C-GLWQ""","""CGLWQ""",42.87529,-85.55405,475,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…","""A2""",false,false,false,false
